# 04 — GEE Landsat LST, OSM Wards & CPCB Calibration

End-to-end accuracy workflow for UrbanYantra:
1. Google Earth Engine Landsat 8 ST_B10 LST
2. OSM neighborhood / administrative ward polygons
3. CPCB station bias calibration (Open-Meteo / ERA5 air temperature reference)
4. Zonal LST statistics per real ward polygon

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

CITY = 'Mumbai'
DATA_MODE = 'gee'  # 'synthetic' if GEE not authenticated

In [ ]:
# Cell 1: GEE setup (run once)
# !pip install earthengine-api geemap osmnx geopandas
import ee
from data_ingestion.gee_client import init_gee
from utils import load_city_config

cfg = load_city_config(ROOT / 'configs' / 'cities.yaml', CITY)
# ee.Authenticate()  # one-time in Colab
init_gee(project=cfg.get('gee_project'))
print('GEE OK — project:', cfg.get('gee_project'))

In [ ]:
# Cell 2: Full pipeline (ingest → features → wards → calibration → train)
from run_pipeline import run_single_city

cfg['data_mode'] = DATA_MODE
cfg['force_refresh'] = True
df = run_single_city(cfg, skip_shap=False)
print(df[['lst_c', 'lst_calibrated_c']].describe())

In [ ]:
# Cell 3: Inspect ward boundaries + calibration
import json
out = ROOT / cfg['output_dir']
cal = json.loads((out / 'calibration' / 'station_calibration.json').read_text())
meta = json.loads((out / 'geojson' / 'ward_boundaries_meta.json').read_text())
print('Calibration bias:', cal['bias_c'], '°C')
for s in cal.get('stations', []):
    print(f"  {s['name']}: air={s['air_temp_c']}°C sat={s['lst_satellite_c']}°C Δ={s['delta_air_minus_lst_c']}°C")
print('Ward sources:', meta)

In [ ]:
# Cell 4: Quick map of ward polygons
import geopandas as gpd
import matplotlib.pyplot as plt

wards = gpd.read_file(out / 'geojson' / 'ward_zonal.geojson')
fig, ax = plt.subplots(figsize=(10, 10))
wards.plot(column='lst_calibrated_c', cmap='RdYlBu_r', legend=True, ax=ax)
for _, r in wards.iterrows():
    c = r.geometry.centroid
    ax.annotate(r['name'], (c.x, c.y), fontsize=7)
ax.set_title(f'{CITY} — calibrated LST by OSM/Voronoi ward')
plt.show()